# MODEL C: Baseline

**high precision** and **low variance**.

**Strategy**:
- **Loss**: Cross-Entropy with Label Smoothing (0.15)
- **Learning Rate**: LOW (encoder: 1e-5, classifier: 5e-4) for stability
- **Dropout**: HIGH (0.2/0.15) for strong regularization
- **Pseudo-labels**: ONLY highest confidence (>0.95), reduced ratio (1.5:1)
- **Training**: LONGER (20 epochs) with more patience (7)
- **Focus**: Stable, conservative predictions - good for ensemble balancing

**Configuration**:
```
Label Smoothing:  0.15 (prevent overconfidence)
Encoder LR:       1e-5  (VERY LOW)
Classifier LR:    5e-4  (VERY LOW)
Hidden Dropout:   0.20  (HIGH)
Attention Drop:   0.15  (HIGH)
Pseudo Conf:      >0.95 (STRICT)
Pseudo Ratio:     1.5:1 (REDUCED)
Epochs:           20    (LONG)
Patience:         7     (PATIENT)
```




In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch
import pickle
import warnings
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
import torch.nn as nn

warnings.filterwarnings('ignore')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Imports loaded")
print(f"Device: {device}")

Imports loaded
Device: cuda


## 1) Load Data with STRICT Pseudo Filtering (conf > 0.95)

In [2]:
print("Loading augmented data...")
augmented = pd.read_csv('data/augmented_training_data.csv', keep_default_na=False, na_values=[''])
real_train = augmented[augmented['split'] == 'train'].copy()
real_val = augmented[augmented['split'] == 'validation'].copy()
real_test = augmented[augmented['split'] == 'test'].copy()
print(f"Real train: {len(real_train)}")

print("\nLoading pseudo-labels with VERY STRICT filtering (conf > 0.95)...")
pseudo = pd.read_csv('data/pseudo_labeled_chunks.csv', keep_default_na=False, na_values=[''])

# ULTRA-STRICT: Only keep confidence > 0.95 AND entropy < 0.2
# Target 1.5:1 ratio (~7000 pseudo for ~4600 real)
pseudo_strict = pseudo[pseudo['confidence'] > 0.95].copy()
print(f"After conf > 0.95: {len(pseudo_strict)}")

# Balanced sampling: ~1200 per class (7200 total)
pseudo_balanced = []
for label in pseudo_strict['pseudo_label'].unique():
    subset = pseudo_strict[pseudo_strict['pseudo_label'] == label]
    subset = subset.nlargest(min(1200, len(subset)), 'confidence')
    pseudo_balanced.append(subset)

pseudo_train = pd.concat(pseudo_balanced, ignore_index=True)
pseudo_train['split'] = 'train'
pseudo_train.rename(columns={'pseudo_label': 'frame_label'}, inplace=True)
print(f"\nPseudo-labeled (STRICT): {len(pseudo_train)}")
print(pseudo_train['frame_label'].value_counts().sort_index())

real_train['sample_weight'] = 1.0
pseudo_train['sample_weight'] = 0.15  # Even lower weight for pseudo
combined_train = pd.concat([real_train, pseudo_train], ignore_index=True)
print(f"\nCombined: {len(combined_train)}")
print(f"Ratio: {len(pseudo_train)/len(real_train):.2f}:1")

Loading augmented data...
Real train: 4606

Loading pseudo-labels with VERY STRICT filtering (conf > 0.95)...
After conf > 0.95: 9000

Pseudo-labeled (STRICT): 7200
frame_label
Conflict         1200
Economic         1200
Human Impact     1200
Moral Value      1200
None             1200
Powerlessness    1200
Name: count, dtype: int64

Combined: 11806
Ratio: 1.56:1


## 2) Encode & Tokenize

In [3]:
with open('data/roberta_label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)
labels = list(label_encoder.classes_)
num_labels = len(labels)

combined_train['label'] = label_encoder.transform(combined_train['frame_label'])
real_val['label'] = label_encoder.transform(real_val['frame_label'])
real_test['label'] = label_encoder.transform(real_test['frame_label'])

# MODERATE class weights (not extreme)
class_weights = torch.FloatTensor([
    1.2,  # Conflict
    0.9,  # Economic
    1.0,  # Human Impact
    1.4,  # Moral Value
    1.1,  # None
    1.4   # Powerlessness
])
print(f"Moderate weights: {class_weights.tolist()}")

tokenizer = RobertaTokenizer.from_pretrained('models/roberta-dapt-dynamic')

def tokenize_function(examples):
    return tokenizer(examples['chunk_text'], truncation=True, max_length=384, padding='max_length')

train_dataset = Dataset.from_pandas(combined_train[['chunk_text','label','sample_weight']])
train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=['chunk_text'])
train_dataset.set_format('torch')

val_dataset = Dataset.from_pandas(real_val[['chunk_text','label']])
val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=['chunk_text'])
val_dataset.set_format('torch')

test_dataset = Dataset.from_pandas(real_test[['chunk_text','label']])
test_dataset = test_dataset.map(tokenize_function, batched=True, remove_columns=['chunk_text'])
test_dataset.set_format('torch')

print(f"Datasets: {len(train_dataset)}, {len(val_dataset)}, {len(test_dataset)}")

Moderate weights: [1.2000000476837158, 0.8999999761581421, 1.0, 1.399999976158142, 1.100000023841858, 1.399999976158142]


Map:   0%|          | 0/11806 [00:00<?, ? examples/s]

Map:   0%|          | 0/683 [00:00<?, ? examples/s]

Map:   0%|          | 0/348 [00:00<?, ? examples/s]

Datasets: 11806, 683, 348


## 3) Label Smoothing Trainer

In [4]:
from dataclasses import dataclass
from typing import Any, Dict, List

@dataclass
class DataCollatorWithSampleWeight(DataCollatorWithPadding):
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        sample_weights = None
        if 'sample_weight' in features[0]:
            sample_weights = torch.FloatTensor([f.pop('sample_weight') for f in features])
        batch = super().__call__(features)
        if sample_weights is not None:
            batch['sample_weight'] = sample_weights
        return batch

class LabelSmoothingTrainer(Trainer):
    def __init__(self, *args, class_weights=None, label_smoothing=0.15, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights.to(self.args.device) if class_weights is not None else None
        self.label_smoothing = label_smoothing
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        sample_weight = inputs.pop('sample_weight', None)
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Cross-Entropy with label smoothing and class weights
        loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights,
            label_smoothing=self.label_smoothing,
            reduction='none'
        )
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        if sample_weight is not None:
            loss = (loss * sample_weight).mean()
        else:
            loss = loss.mean()
        
        return (loss, outputs) if return_outputs else loss

print("Label Smoothing Trainer (0.15) defined")

Label Smoothing Trainer (0.15) defined


## 4) Load Model with HIGH Dropout

In [5]:
print("Loading model with HIGH dropout for regularization...")
model = RobertaForSequenceClassification.from_pretrained(
    'models/roberta-dapt-dynamic',
    num_labels=num_labels,
    attention_probs_dropout_prob=0.15,  # HIGH (was 0.05)
    hidden_dropout_prob=0.20,           # HIGH (was 0.10)
    ignore_mismatched_sizes=True
)
model = model.to(device)
print("Model loaded with HIGH dropout (0.15/0.20)")

Loading model with HIGH dropout for regularization...
Model loaded with HIGH dropout (0.15/0.20)


## 5) Optimizer with VERY LOW Learning Rates

In [6]:
encoder_params = [p for n, p in model.named_parameters() if 'classifier' not in n]
classifier_params = [p for n, p in model.named_parameters() if 'classifier' in n]

encoder_lr = 1e-5   # VERY LOW (was 2-3e-5)
classifier_lr = 5e-4  # VERY LOW (was 1-5e-3)

optimizer = AdamW([
    {'params': encoder_params, 'lr': encoder_lr},
    {'params': classifier_params, 'lr': classifier_lr}
], weight_decay=0.05)

# Standard cosine schedule (no restarts - conservative)
num_training_steps = len(train_dataset) // 8 // 4 * 20  # 20 epochs
num_warmup_steps = int(0.1 * num_training_steps)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)

print(f"CONSERVATIVE LRs: encoder={encoder_lr}, classifier={classifier_lr}")

CONSERVATIVE LRs: encoder=1e-05, classifier=0.0005


## 6) Metrics

In [7]:
def compute_metrics(eval_pred):
    logits, labels_np = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels_np, preds)
    f1_macro = f1_score(labels_np, preds, average='macro')
    f1_weighted = f1_score(labels_np, preds, average='weighted')
    _, _, f1_per_class, _ = precision_recall_fscore_support(labels_np, preds, average=None, zero_division=0)
    metrics = {'accuracy': acc, 'f1_macro': f1_macro, 'f1_weighted': f1_weighted}
    for i, label in enumerate(labels):
        metrics[f'f1_{label}'] = f1_per_class[i]
    return metrics

print("Metrics defined")

Metrics defined


## 7) Training (20 Epochs, Patience=7)

In [8]:
training_args = TrainingArguments(
    output_dir='models/roberta-ensemble-model-C',
    num_train_epochs=20,  # LONG training
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=4,
    weight_decay=0.05,
    max_grad_norm=5.0,
    eval_strategy='steps',
    eval_steps=50,  # Evaluate every 50 steps!
    save_strategy='steps',
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    logging_steps=50,
    report_to='none',
    seed=42
)

data_collator = DataCollatorWithSampleWeight(tokenizer=tokenizer, padding=True)

trainer = LabelSmoothingTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=7)],  # Patient!
    data_collator=data_collator,
    optimizers=(optimizer, scheduler),
    class_weights=class_weights,
    label_smoothing=0.15
)

print("=" * 80)
print("MODEL C: CONSERVATIVE BASELINE")
print("=" * 80)
print("Strategy: Stability & precision over aggression")
print("Label Smoothing: 0.15")
print("Dropout: HIGH (0.15/0.20)")
print("LR: VERY LOW (1e-5 / 5e-4)")
print("Pseudo: STRICT (conf>0.95, ratio 1.5:1)")
print("Target: F1=0.70 with low variance")
print("=" * 80)

train_result = trainer.train()
print("\nTraining complete!")

MODEL C: CONSERVATIVE BASELINE
Strategy: Stability & precision over aggression
Label Smoothing: 0.15
Dropout: HIGH (0.15/0.20)
LR: VERY LOW (1e-5 / 5e-4)
Pseudo: STRICT (conf>0.95, ratio 1.5:1)
Target: F1=0.70 with low variance


Step,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,F1 Conflict,F1 Economic,F1 Human impact,F1 Moral value,F1 None,F1 Powerlessness
50,1.381100,2.494666,0.688141,0.685520,0.686298,0.672646,0.815385,0.718182,0.612069,0.727273,0.567568
100,1.119900,1.743124,0.696925,0.698391,0.699368,0.654028,0.841667,0.745455,0.620408,0.735294,0.593496
150,0.875700,1.539706,0.680820,0.682468,0.683373,0.619289,0.830508,0.719298,0.596899,0.720812,0.608000
200,0.822600,1.517750,0.677892,0.680689,0.681538,0.605769,0.837607,0.711111,0.591760,0.726316,0.611570
250,0.793500,1.475937,0.682284,0.680364,0.681346,0.568528,0.836653,0.724771,0.605578,0.717172,0.629482
300,0.778600,1.448821,0.696925,0.696645,0.697532,0.643902,0.814516,0.737778,0.632911,0.717949,0.632812
350,0.761600,1.461524,0.692533,0.692042,0.693007,0.601036,0.840000,0.739726,0.624000,0.729167,0.618321
400,0.747900,1.468857,0.673499,0.673861,0.674913,0.626609,0.817460,0.710900,0.605578,0.698925,0.583691
450,0.734000,1.442870,0.691069,0.692409,0.693380,0.644550,0.822034,0.728111,0.622047,0.712195,0.625514



Training complete!


## 8) Evaluate & Save

In [9]:
print("\nValidation...")
val_results = trainer.evaluate(eval_dataset=val_dataset)
print(val_results)

print("\nTest...")
test_results = trainer.evaluate(eval_dataset=test_dataset)
print(test_results)

os.makedirs('models/roberta-ensemble-model-C', exist_ok=True)
trainer.save_model('models/roberta-ensemble-model-C')
tokenizer.save_pretrained('models/roberta-ensemble-model-C')

os.makedirs('results', exist_ok=True)
with open('results/model_C_metrics.json', 'w') as f:
    json.dump({
        'model': 'C - Conservative Baseline',
        'strategy': 'Label smoothing + HIGH dropout + LOW LR',
        'label_smoothing': 0.15,
        'dropout': [0.15, 0.20],
        'lr': [1e-5, 5e-4],
        'weights': class_weights.tolist(),
        'val': val_results,
        'test': test_results,
        'time': train_result.metrics['train_runtime']
    }, f, indent=2)

print("\n" + "="*80)
print("MODEL C RESULTS:")
print(f"Val F1: {val_results['eval_f1_macro']:.4f}")
print(f"Test F1: {test_results['eval_f1_macro']:.4f}")
print("\nPer-Class F1:")
for label in labels:
    print(f"  {label:20s}: {test_results[f'eval_f1_{label}']:.4f}")
print("="*80)


Validation...


{'eval_loss': 1.7431243658065796, 'eval_accuracy': 0.6969253294289898, 'eval_f1_macro': 0.6983913106686473, 'eval_f1_weighted': 0.699368235125792, 'eval_f1_Conflict': 0.6540284360189573, 'eval_f1_Economic': 0.8416666666666667, 'eval_f1_Human Impact': 0.7454545454545455, 'eval_f1_Moral Value': 0.6204081632653061, 'eval_f1_None': 0.7352941176470589, 'eval_f1_Powerlessness': 0.5934959349593496, 'eval_runtime': 25.1419, 'eval_samples_per_second': 27.166, 'eval_steps_per_second': 0.875, 'epoch': 1.2195121951219512}

Test...
{'eval_loss': 1.6574959754943848, 'eval_accuracy': 0.6896551724137931, 'eval_f1_macro': 0.6921931408884084, 'eval_f1_weighted': 0.6921931408884084, 'eval_f1_Conflict': 0.7058823529411765, 'eval_f1_Economic': 0.7567567567567568, 'eval_f1_Human Impact': 0.7207207207207207, 'eval_f1_Moral Value': 0.6530612244897959, 'eval_f1_None': 0.7027027027027027, 'eval_f1_Powerlessness': 0.6140350877192983, 'eval_runtime': 12.4375, 'eval_samples_per_second': 27.98, 'eval_steps_per_seco